# GSE193531 — Step 1: Raw Count Matrix Construction and Cell-Level QC

Builds a single AnnData object from the dense UMI count matrix and the cell-level
metadata CSV, computes QC metrics, runs per-sample doublet detection (Scrublet),
applies hybrid (adaptive + fixed-floor) QC thresholds, and writes the filtered
AnnData object used by every later step in this pipeline.

**Input** (place under `data/raw/GSE193531/`, downloaded from GEO accession GSE193531):
- `GSE193531_umi-count-matrix.csv` — genes (rows) x cells (columns), float-valued
- `GSE193531_cell-level-metadata.csv` — per-cell `sample_ID` / `disease_stage`

**Output:**
- `data/processed/GSE193531/GSE193531_raw_counts.h5ad`
- `data/processed/GSE193531/GSE193531_filtered.h5ad` (production object for step 2)
- `results/GSE193531/metadata/GSE193531_cell_manifest.csv`
- `results/GSE193531/qc/GSE193531_cell_qc.csv`, `..._cell_qc_filtered.csv`, `..._filter_report.csv`, `..._stage_filter_summary.csv`

This is a minimal, single-cohort notebook: the exploratory-only cells from the
original project notebook (a single-sample deep-dive comparison, and an alternate
"doublets-only-removed" save that is explicitly not used downstream) are omitted.
Everything below is the production path that `GSE193531_filtered.h5ad` is actually
built from.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation

#### Directory layout and required files

In [ ]:
PROJECT = Path("../")
COHORT = "GSE193531"

RAW_DIR = PROJECT / "data" / "raw" / COHORT
PROCESSED_DIR = PROJECT / "data" / "processed" / COHORT
RESULTS_DIR = PROJECT / "results" / COHORT

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "metadata").mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "qc").mkdir(parents=True, exist_ok=True)

COUNT_MATRIX_FILE = RAW_DIR / f"{COHORT}_umi-count-matrix.csv"
CELL_METADATA_FILE = RAW_DIR / f"{COHORT}_cell-level-metadata.csv"

if not COUNT_MATRIX_FILE.exists():
    raise FileNotFoundError(COUNT_MATRIX_FILE)
if not CELL_METADATA_FILE.exists():
    raise FileNotFoundError(CELL_METADATA_FILE)

# Fixed sanity guardrails (backstops, not the primary QC method).
MIN_COUNTS = 500
MIN_GENES = 200
MAX_PCT_MITO = 15.0

# Diagnostic-only threshold (NOT a removal guardrail): cells above this are
# flagged for manual review, not automatically removed. Doublet removal is
# driven entirely by Scrublet.
MAX_GENES_DIAGNOSTIC = 6500

## 1. Load and Harmonize Cell-Level Metadata

### Processing
The cell-level metadata file already provides `sample_ID` and `disease_stage` per
cell. The stage label (`NBM`, `MGUS`, `SMM`, `MM`) is mapped to the standardized
Healthy/MGUS/SMM/MM classes. `patient_id` is set equal to `sample_id`, since this
cohort is one bone marrow aspirate per patient.

In [ ]:
STAGE_MAP = {"NBM": "Healthy", "MGUS": "MGUS", "SMM": "SMM", "MM": "MM"}

cell_metadata = pd.read_csv(CELL_METADATA_FILE)
cell_metadata = cell_metadata.rename(
    columns={"index": "cell_id", "sample_ID": "sample_id", "disease_stage": "stage"}
)

unmapped = set(cell_metadata["stage"].dropna().unique()) - set(STAGE_MAP)
if unmapped:
    raise ValueError(f"Unrecognized stage label(s): {sorted(unmapped)}")

cell_metadata["clinical_stage_standard"] = cell_metadata["stage"].map(STAGE_MAP)
cell_metadata["patient_id"] = cell_metadata["sample_id"]

if cell_metadata["cell_id"].duplicated().any():
    raise ValueError("Duplicate cell_id values in cell metadata.")

cell_metadata = cell_metadata.set_index("cell_id")
cell_metadata.head()

## Build the Count Matrix from the Dense UMI Matrix

In [ ]:
header_columns = pd.read_csv(COUNT_MATRIX_FILE, nrows=0).columns.tolist()
data_dtype = {column: np.float32 for column in header_columns[1:]}

counts_genes_by_cells = pd.read_csv(COUNT_MATRIX_FILE, index_col=0, dtype=data_dtype)

X = csr_matrix(counts_genes_by_cells.values.T)

adata = sc.AnnData(
    X=X,
    obs=pd.DataFrame(index=counts_genes_by_cells.columns.astype(str)),
    var=pd.DataFrame(index=counts_genes_by_cells.index.astype(str)),
)
adata.var_names_make_unique()

del counts_genes_by_cells

missing_metadata = set(adata.obs_names) - set(cell_metadata.index)
missing_matrix = set(cell_metadata.index) - set(adata.obs_names)

if missing_metadata:
    raise ValueError(f"{len(missing_metadata)} cells in the count matrix have no metadata.")
if missing_matrix:
    raise ValueError(f"{len(missing_matrix)} cells in the metadata have no count matrix row.")

adata.obs = cell_metadata.loc[
    adata.obs_names, ["sample_id", "patient_id", "clinical_stage_standard"]
].copy()

assert adata.obs_names.is_unique
assert adata.obs.notna().all().all()

count_matrix_path = PROCESSED_DIR / f"{COHORT}_raw_counts.h5ad"
adata.write(count_matrix_path)

cell_manifest = adata.obs.reset_index(names="cell_id")
cell_manifest["cohort"] = COHORT
manifest_path = RESULTS_DIR / "metadata" / f"{COHORT}_cell_manifest.csv"
cell_manifest.to_csv(manifest_path, index=False)

print(f"Count matrix: {adata.n_obs:,} cells x {adata.n_vars:,} genes -> {count_matrix_path}")
print(f"Cell manifest -> {manifest_path}")

## Cell-Level QC Metrics

`complexity` = log10(detected genes) / log10(total counts) (Ilicic et al. 2016;
scater/OSCA "novelty score" convention) — **not** the raw genes/counts ratio.

In [ ]:
symbol = pd.Series(adata.var_names.astype(str)).str.upper()

gene_masks = {
    "mito": symbol.str.startswith("MT-").to_numpy(),
    "ribo": (symbol.str.startswith("RPS") | symbol.str.startswith("RPL")).to_numpy(),
    "hb": symbol.str.startswith(("HBA", "HBB", "HBD", "HBE", "HBG", "HBZ")).to_numpy(),
    "ig": symbol.str.startswith(("IGH", "IGK", "IGL")).to_numpy(),
}

X = adata.X
n_counts = np.asarray(X.sum(axis=1)).ravel()
n_genes = np.asarray((X > 0).sum(axis=1)).ravel()


def subset_counts(mask):
    return np.asarray(X[:, mask].sum(axis=1)).ravel()


denominator = np.where(n_counts > 0, n_counts, np.nan)

qc = pd.DataFrame(index=adata.obs_names)
qc["n_counts"] = n_counts
qc["n_genes"] = n_genes
qc["pct_mito"] = 100 * subset_counts(gene_masks["mito"]) / denominator
qc["pct_ribo"] = 100 * subset_counts(gene_masks["ribo"]) / denominator
qc["pct_hb"] = 100 * subset_counts(gene_masks["hb"]) / denominator
qc["pct_ig"] = 100 * subset_counts(gene_masks["ig"]) / denominator
with np.errstate(divide="ignore", invalid="ignore"):
    qc["complexity"] = np.log10(qc["n_genes"]) / np.log10(qc["n_counts"])

qc = qc.join(adata.obs[["sample_id", "patient_id", "clinical_stage_standard"]])

qc_path = RESULTS_DIR / "qc" / f"{COHORT}_cell_qc.csv"
qc.to_csv(qc_path)

print(f"QC metrics for {len(qc):,} cells -> {qc_path}")
qc[["n_counts", "n_genes", "pct_mito", "pct_ribo", "pct_hb", "pct_ig", "complexity"]].describe()

## Doublet Detection (Scrublet, per Sample)

Doublet detection is run independently within each `sample_id`.

In [ ]:
sc.pp.scrublet(adata, batch_key="sample_id", random_state=0)

doublets = adata.obs[["doublet_score", "predicted_doublet"]].copy()
qc = qc.drop(columns=["doublet_score", "predicted_doublet"], errors="ignore").join(doublets)
qc.to_csv(qc_path)

n_predicted = int(qc["predicted_doublet"].fillna(False).sum())
print(f"Predicted doublets: {n_predicted:,} / {len(qc):,} cells ({n_predicted / len(qc):.2%})")

qc.groupby("sample_id", observed=True)[["doublet_score", "predicted_doublet"]].agg(
    median_doublet_score=("doublet_score", "median"),
    n_predicted_doublets=("predicted_doublet", "sum"),
).sort_values("n_predicted_doublets", ascending=False).head(10)

## Define Hybrid Sample-Specific QC Thresholds

- Low `n_counts`: adaptive lower outlier threshold in log1p space (median − 3 MAD), with a fixed floor of 500.
- Low `n_genes`: same, with a fixed floor of 200.
- High `pct_mito`: adaptive upper outlier threshold, capped at 15%.

Final thresholds are hybrid: `final lower = max(adaptive lower, fixed floor)`,
`final upper = min(adaptive upper, fixed ceiling)`. No cells are removed in this
step.

In [ ]:
def sample_summary(df):
    meta = df[["sample_id", "patient_id", "clinical_stage_standard"]].drop_duplicates().set_index("sample_id")
    agg = df.groupby("sample_id", observed=True).agg(
        n_cells=("n_counts", "size"),
        median_n_counts=("n_counts", "median"),
        median_n_genes=("n_genes", "median"),
        median_pct_mito=("pct_mito", "median"),
        median_pct_ribo=("pct_ribo", "median"),
        median_pct_hb=("pct_hb", "median"),
        median_pct_ig=("pct_ig", "median"),
        median_complexity=("complexity", "median"),
    )
    return agg.join(meta)


def adaptive_sample_thresholds(df, nmads=3):
    # Per-sample adaptive QC thresholds. Lower thresholds for n_counts/n_genes
    # are in log1p space. The upper threshold for pct_mito is on the percentage
    # scale. Final thresholds are hybrid: lower = max(adaptive, fixed floor),
    # upper = min(adaptive, fixed ceiling).
    rows = []
    for sample_id, group in df.groupby("sample_id", observed=True):
        for metric, fixed_floor in [("n_counts", MIN_COUNTS), ("n_genes", MIN_GENES)]:
            values = np.log1p(group[metric].to_numpy(dtype=float))
            med = np.nanmedian(values)
            mad = median_abs_deviation(values, scale="normal", nan_policy="omit")
            if not np.isfinite(mad) or mad == 0:
                mad = np.nanstd(values)
            if not np.isfinite(mad) or mad == 0:
                mad = 0.0
            adaptive_low = max(np.expm1(med - nmads * mad), 0.0)
            final_low = max(adaptive_low, fixed_floor)
            rows.append({
                "sample_id": sample_id, "metric": metric,
                "adaptive_low": adaptive_low, "fixed_floor": fixed_floor, "final_low": final_low,
                "adaptive_high": np.nan, "fixed_ceiling": np.nan, "final_high": np.nan,
            })

        values = group["pct_mito"].to_numpy(dtype=float)
        med = np.nanmedian(values)
        mad = median_abs_deviation(values, scale="normal", nan_policy="omit")
        if not np.isfinite(mad) or mad == 0:
            mad = np.nanstd(values)
        if not np.isfinite(mad) or mad == 0:
            mad = 0.0
        adaptive_high = med + nmads * mad
        final_high = min(adaptive_high, MAX_PCT_MITO)
        rows.append({
            "sample_id": sample_id, "metric": "pct_mito",
            "adaptive_low": np.nan, "fixed_floor": np.nan, "final_low": np.nan,
            "adaptive_high": adaptive_high, "fixed_ceiling": MAX_PCT_MITO, "final_high": final_high,
        })
    return pd.DataFrame(rows)


def apply_sample_thresholds(qc, thresholds):
    qc = qc.copy()
    low_table = thresholds.pivot(index="sample_id", columns="metric", values="final_low")
    high_table = thresholds.pivot(index="sample_id", columns="metric", values="final_high")
    qc["sample_n_counts_low"] = qc["sample_id"].map(low_table["n_counts"])
    qc["sample_n_genes_low"] = qc["sample_id"].map(low_table["n_genes"])
    qc["sample_pct_mito_high"] = qc["sample_id"].map(high_table["pct_mito"])
    return qc


stage_order = ["Healthy", "MGUS", "SMM", "MM"]
qc["clinical_stage_standard"] = pd.Categorical(qc["clinical_stage_standard"], categories=stage_order, ordered=True)

summary = sample_summary(qc)
summary.sort_values("n_cells", ascending=False)

In [ ]:
sample_thresholds = adaptive_sample_thresholds(qc)
print(f"Hybrid QC thresholds: adaptive per sample + fixed sanity guardrails")
print(f"MIN_COUNTS={MIN_COUNTS}, MIN_GENES={MIN_GENES}, MAX_PCT_MITO={MAX_PCT_MITO}")
sample_thresholds

## Flag Cells for QC Removal

A cell is marked for removal when:

`flag_any_qc = low_counts OR low_genes OR high_mito OR predicted_doublet`

Hemoglobin percentage, immunoglobulin percentage, library complexity, and cells
with more than `MAX_GENES_DIAGNOSTIC` (6,500) detected genes remain diagnostic
metrics, not automatic removal criteria.

In [ ]:
qc = apply_sample_thresholds(qc, sample_thresholds)

for col in ["sample_n_counts_low", "sample_n_genes_low", "sample_pct_mito_high"]:
    qc[col] = pd.to_numeric(qc[col], errors="coerce")

qc["flag_low_counts"] = qc["n_counts"] < qc["sample_n_counts_low"]
qc["flag_low_genes"] = qc["n_genes"] < qc["sample_n_genes_low"]
qc["flag_high_mito"] = qc["pct_mito"] > qc["sample_pct_mito_high"]
qc["flag_predicted_doublet"] = qc["predicted_doublet"].fillna(False).astype(bool)

# Diagnostic only -- not used for automatic removal.
qc["flag_high_hb"] = qc["pct_hb"] > 2.0
qc["flag_low_complexity"] = qc["complexity"] < 0.80
qc["flag_high_genes"] = qc["n_genes"] > MAX_GENES_DIAGNOSTIC

remove_flags = ["flag_low_counts", "flag_low_genes", "flag_high_mito", "flag_predicted_doublet"]
diagnostic_flags = ["flag_high_hb", "flag_low_complexity", "flag_high_genes"]

qc["flag_any_qc"] = qc[remove_flags].any(axis=1)
qc.to_csv(qc_path)

print("Cells flagged by each removal criterion (non-exclusive):")
display(qc[remove_flags].sum().rename("n_cells_flagged").to_frame())

print(f"\nDiagnostic flags (not used for removal); flag_high_genes threshold = {MAX_GENES_DIAGNOSTIC} genes:")
display(qc[diagnostic_flags].sum().rename("n_cells_flagged").to_frame())

print("\nCells flagged for removal by sample:")
qc.groupby("sample_id", observed=True)["flag_any_qc"].agg(
    n_cells="size", flagged="sum", flagged_fraction="mean"
).sort_values("flagged_fraction", ascending=False)

### Final Per-Sample Filter Report

In [ ]:
report = qc.groupby(["sample_id", "patient_id", "clinical_stage_standard"], observed=True).agg(
    n_cells=("n_counts", "size"),
    median_n_counts=("n_counts", "median"),
    median_n_genes=("n_genes", "median"),
    median_pct_mito=("pct_mito", "median"),
    median_pct_hb=("pct_hb", "median"),
    median_pct_ig=("pct_ig", "median"),
    median_complexity=("complexity", "median"),
    flagged_cells=("flag_any_qc", "sum"),
    flagged_fraction=("flag_any_qc", "mean"),
    n_predicted_doublets=("predicted_doublet", lambda x: int(x.fillna(False).sum())),
).reset_index()


def _suggest_action(row):
    if row["flagged_fraction"] >= 0.5 and row["n_cells"] >= 50:
        return "review_strong_exclude_candidate"
    if row["flagged_fraction"] >= 0.25:
        return "review"
    if row["n_predicted_doublets"] >= 0.05 * max(1, row["n_cells"]):
        return "review_doublets"
    return "keep"


report["suggested_action"] = report.apply(_suggest_action, axis=1)

report_path = RESULTS_DIR / "qc" / f"{COHORT}_filter_report.csv"
report.to_csv(report_path, index=False)

stage_report = report.groupby("clinical_stage_standard", observed=True).agg(
    n_samples=("sample_id", "nunique"),
    total_cells=("n_cells", "sum"),
    mean_flagged_fraction=("flagged_fraction", "mean"),
).reset_index()
stage_report_path = RESULTS_DIR / "qc" / f"{COHORT}_stage_filter_summary.csv"
stage_report.to_csv(stage_report_path, index=False)

print("Saved per-sample filter report ->", report_path)
print("Saved per-stage summary ->", stage_report_path)
print("\nOverall flagged cells (any criterion):", int(qc["flag_any_qc"].sum()), "of", len(qc),
      f'({qc["flag_any_qc"].mean():.2%})')

display(report.sort_values("flagged_fraction", ascending=False).head(30))
print("\nStage summary:")
display(stage_report)

## Apply Final Cell-Level QC Filtering

Cells are removed only when `flag_any_qc == True`. The QC index is checked
against `adata.obs_names` first to make sure the same cells are being filtered.

In [ ]:
remove_mask = qc["flag_any_qc"].astype(bool)

if not remove_mask.index.equals(adata.obs_names):
    raise ValueError("QC table index does not match AnnData cell order. Do not filter until this is resolved.")

n_remove = int(remove_mask.sum())
n_total = len(qc)
print(f"Removing {n_remove:,} / {n_total:,} cells ({n_remove / n_total:.2%})")

keep_cells = qc.index[~remove_mask]
adata_filt = adata[keep_cells].copy()

if adata_filt.n_obs != len(keep_cells):
    raise RuntimeError("Unexpected mismatch after filtering.")

output_path = PROCESSED_DIR / f"{COHORT}_filtered.h5ad"
adata_filt.write(output_path)

qc.loc[keep_cells].to_csv(RESULTS_DIR / "qc" / f"{COHORT}_cell_qc_filtered.csv", index=True)

print(f"Saved filtered AnnData: {output_path}")
print(f"Cells retained: {adata_filt.n_obs:,} / {n_total:,}")